In [ ]:
# --- repo bootstrap: make src/ + this domain's config importable, run from repo root ---
# domains/credit_risk/ is mounted only in the credit-risk containers, so the
# domain-agnostic stages physically cannot import domain settings.
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
sys.path.insert(0, str(_ROOT / 'domains' / 'credit_risk'))
os.chdir(_ROOT)

# Aave V3.1 — model dataset (daily credit-risk panel)

Assembles the modeling table: the 2h protocol panel aggregated to daily grain
(events/flows sum, ratios REBUILT as ratio-of-sums from the daily flows — not
the mean of 12 bucket ratios), joined with the dense 24h liquidation +
user-account risk frames, restricted to the credit-risk feature set, with
forward-looking stress/volume targets. Exports `DF_model_dataset_24h.csv`.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path
from IPython.display import display

import model_config as cfg
import model_dataset as mds
import adv_validation as adv

DATA_DIR = Path("transformed_data")
PREVIEW_ROWS = 5

In [3]:
DF_common_final_1 = pd.read_csv(DATA_DIR / "DF_common_final_1.csv")
DF_liq_24h = pd.read_csv(DATA_DIR / "DF_liq_features_24h.csv")
DF_user_24h = pd.read_csv(DATA_DIR / "DF_user_features_24h.csv")

for name, df in [("DF_common_final_1", DF_common_final_1),
                 ("DF_liq_features_24h", DF_liq_24h),
                 ("DF_user_features_24h", DF_user_24h)]:
    print(f" {name}: {df.shape[0]} rows x {df.shape[1]} cols")
display(DF_user_24h.head(PREVIEW_ROWS))

 DF_common_final_1: 4368 rows x 61 cols
 DF_liq_features_24h: 364 rows x 24 cols
 DF_user_features_24h: 364 rows x 21 cols


,time_bucket,avg_total_collateral_base,avg_total_debt_base,avg_available_borrows_base,avg_current_liquidation_threshold,avg_ltv,sampled_user_count,account_data_call_count,user_state_observed,user_state_age,...,collateralization_ratio,borrow_capacity_utilization,remaining_borrow_capacity,risk_buffer,ltv_utilization,distance_to_liquidation,debt_expansion_ratio,capital_efficiency,overcollateral_margin,has_sampled_borrow_position
0,2025-04-01 00:00:00.000 UTC,1.011393e+08,4.153081e+07,4.166146e+07,0.837500,0.812500,4,4,True,0.0,...,2.435283,0.499215,0.411922,0.025000,0.970149,0.029851,0.410630,2.923246,5.960846e+07,1
1,2025-04-02 00:00:00.000 UTC,5.239258e+07,3.211520e+07,1.660917e+07,0.864650,0.828583,4,4,True,0.0,...,1.631395,0.659120,0.317014,0.036067,0.958288,0.041712,0.612972,5.026781,2.027738e+07,1
2,2025-04-03 00:00:00.000 UTC,3.098535e+07,9.922407e+06,1.458708e+07,0.794367,0.769311,14,29,True,0.0,...,3.122766,0.404839,0.470774,0.025056,0.968458,0.031542,0.320229,7.329376,2.106295e+07,1
3,2025-04-04 00:00:00.000 UTC,5.013800e+07,4.042213e+07,5.059327e+06,0.835460,0.800625,7,9,True,0.0,...,1.240360,0.888761,0.100908,0.034835,0.958304,0.041696,0.806217,4.342659,9.715877e+06,1
4,2025-04-05 00:00:00.000 UTC,9.303986e+07,5.738218e+07,2.384002e+07,0.915000,0.893750,4,8,True,0.0,...,1.621407,0.706484,0.256234,0.021250,0.976776,0.023224,0.616748,1.376022,3.565768e+07,1


In [4]:
# 2h -> daily rollup; ratio columns recomputed as ratio-of-sums inside
df_daily = mds.build_daily_panel(DF_common_final_1)
amap = mds.daily_agg_map([c for c in DF_common_final_1.columns if c != "time_bucket"])
print(f" {df_daily.shape[0]} rows x {df_daily.shape[1]} cols")
print(f" agg map: {sum(1 for v in amap.values() if v == 'sum')} sum, "
      f"{sum(1 for v in amap.values() if v == 'mean')} mean")
display(df_daily.head(PREVIEW_ROWS))

 recomputed 7 daily ratios as ratio-of-sums
 364 rows x 61 cols
 agg map: 36 sum, 24 mean


,time_bucket,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth,borrow_tx_count,...,avg_flashloan_size_eth,flashloan_fee_rate,flashloan_usage_intensity,flashloan_user_activity,variable_debt_flashloan_ratio,no_debt_flashloan_ratio,user_activity,protocol_turnover_usd,protocol_turnover_eth,leverage_indicator
0,2025-04-01 00:00:00.000 UTC,1414,1228,687,646,3.909576e+09,2.084155e+06,3.790291e+09,2.019508e+06,861.0,...,38.289584,0.000206,0.110330,1.502217,0.285285,0.714715,1459.0,8.324287e+09,4.436947e+06,0.075623
1,2025-04-02 00:00:00.000 UTC,1735,1537,792,684,3.084144e+09,1.651553e+06,3.128765e+09,1.676288e+06,895.0,...,33.268024,0.000207,0.150570,1.922853,0.236742,0.763258,1622.0,6.791078e+09,3.636095e+06,0.085394
2,2025-04-03 00:00:00.000 UTC,1924,1580,952,799,2.259890e+09,1.257598e+06,2.157376e+09,1.201185e+06,868.0,...,19.855126,0.000210,0.141117,1.634416,0.245748,0.754252,1849.0,4.922687e+09,2.739975e+06,0.100493
3,2025-04-04 00:00:00.000 UTC,1524,1335,752,673,2.644911e+09,1.466144e+06,2.548904e+09,1.413079e+06,741.0,...,115.050206,0.000171,0.110375,1.638949,0.255156,0.744844,1473.0,5.665841e+09,3.140234e+06,0.082321
4,2025-04-05 00:00:00.000 UTC,1030,903,554,537,1.843859e+09,1.020556e+06,1.835969e+09,1.016933e+06,519.0,...,39.484612,0.000214,0.094900,1.520289,0.332204,0.667796,1055.0,3.956460e+09,2.190817e+06,0.069433


In [ ]:
# check: summed flows and rebuilt ratios both remain well-formed after the rollup
sum_cols = [c for c, v in amap.items() if v == "sum"][:8]
# all 7 rebuilt ratios: the 4 simple num/den ratios PLUS the 3 share ratios
ratio_cols = [c for c in (*cfg.RATIO_RECIPES, *cfg.SHARE_RECIPES) if c in df_daily.columns]
temp = adv.statistical_validation(df_daily, columns=sum_cols + ratio_cols, save=False)
display(temp[["column", "null_pct", "zero_pct", "mean", "cv", "p95"]])

In [6]:
DF_model_full = mds.join_model_frames(df_daily, DF_liq_24h, DF_user_24h)
# liquidation ratios are NA on zero-liquidation days -> structural zeros (E29 precedent)
DF_model_full = mds.fill_conditional_zeros(DF_model_full)
print(f" {DF_model_full.shape[0]} rows x {DF_model_full.shape[1]} cols")
assert DF_model_full.shape[0] >= 360, "daily join lost too many rows"
display(DF_model_full.head(PREVIEW_ROWS))

 inner join: 364 rows kept, 0 dropped (daily 364 / liq 364 / user 364)
 364 rows x 100 cols


,time_bucket,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth,borrow_tx_count,...,borrow_capacity_utilization,remaining_borrow_capacity,risk_buffer,ltv_utilization,distance_to_liquidation,debt_expansion_ratio,capital_efficiency,overcollateral_margin,has_sampled_borrow_position,has_liquidation
0,2025-04-01 00:00:00.000 UTC,1414,1228,687,646,3.909576e+09,2.084155e+06,3.790291e+09,2.019508e+06,861.0,...,0.499215,0.411922,0.025000,0.970149,0.029851,0.410630,2.923246,5.960846e+07,1,1
1,2025-04-02 00:00:00.000 UTC,1735,1537,792,684,3.084144e+09,1.651553e+06,3.128765e+09,1.676288e+06,895.0,...,0.659120,0.317014,0.036067,0.958288,0.041712,0.612972,5.026781,2.027738e+07,1,1
2,2025-04-03 00:00:00.000 UTC,1924,1580,952,799,2.259890e+09,1.257598e+06,2.157376e+09,1.201185e+06,868.0,...,0.404839,0.470774,0.025056,0.968458,0.031542,0.320229,7.329376,2.106295e+07,1,1
3,2025-04-04 00:00:00.000 UTC,1524,1335,752,673,2.644911e+09,1.466144e+06,2.548904e+09,1.413079e+06,741.0,...,0.888761,0.100908,0.034835,0.958304,0.041696,0.806217,4.342659,9.715877e+06,1,1
4,2025-04-05 00:00:00.000 UTC,1030,903,554,537,1.843859e+09,1.020556e+06,1.835969e+09,1.016933e+06,519.0,...,0.706484,0.256234,0.021250,0.976776,0.023224,0.616748,1.376022,3.565768e+07,1,1


In [7]:
# null profile of the joined frame (nothing should exceed the known avg_* residuals)
temp_1 = adv.statistical_validation(DF_model_full, save=False)
display(temp_1.loc[temp_1["null_pct"] > 0,
                   ["column", "null_pct", "zero_pct", "mean", "cv", "p95"]])

,column,null_pct,zero_pct,mean,cv,p95
87,collateralization_ratio,1.0989,0.0,2.208318,0.511423,4.229168


In [8]:
# register the tiered credit-risk weights in the shared priority map (single scheme)
priority_df = mds.sync_priorities()
n_weighted = int(priority_df["column"].isin(list(cfg.CREDIT_RISK_WEIGHTS)).sum())
print(f" priority map: {len(priority_df)} columns, {n_weighted} carrying "
      f"tiered credit-risk weights")

 priority map: 98 columns, 53 carrying tiered credit-risk weights


In [9]:
feat_cols = mds.select_credit_risk_columns(DF_model_full)
print(feat_cols)

 selected 53 credit-risk feature columns
['liquidation_tx_count', 'unique_liquidated_users', 'unique_liquidators', 'as_collateral_tx_count', 'as_debt_tx_count', 'liquidated_collateral_value_usd', 'liquidated_collateral_value_eth', 'liquidation_debt_covered_value_usd', 'liquidation_debt_covered_value_eth', 'liquidation_rate', 'liquidation_volume_ratio_eth', 'liquidation_severity_usd', 'liquidation_severity_eth', 'avg_liquidation_debt_usd', 'avg_liquidation_debt_eth', 'liquidator_concentration', 'liquidation_user_ratio', 'market_stress_index_usd', 'market_stress_index_eth', 'has_liquidation', 'avg_total_collateral_base', 'avg_total_debt_base', 'avg_available_borrows_base', 'avg_current_liquidation_threshold', 'avg_ltv', 'borrow_capacity_utilization', 'remaining_borrow_capacity', 'risk_buffer', 'ltv_utilization', 'distance_to_liquidation', 'debt_expansion_ratio', 'capital_efficiency', 'overcollateral_margin', 'has_sampled_borrow_position', 'borrow_tx_count', 'unique_borrowers', 'borrow_am

In [10]:
# forward targets: next-day USD volume + forward-max per horizon + log1p regression target
keep = ["time_bucket"] + feat_cols
DF_model = mds.add_forward_targets(DF_model_full[keep])
print(f" {DF_model.shape[0]} rows x {DF_model.shape[1]} cols")
display(DF_model.tail(PREVIEW_ROWS))

 dropped 7 trailing rows with incomplete forward windows
 357 rows x 59 cols


,time_bucket,liquidation_tx_count,unique_liquidated_users,unique_liquidators,as_collateral_tx_count,as_debt_tx_count,liquidated_collateral_value_usd,liquidated_collateral_value_eth,liquidation_debt_covered_value_usd,liquidation_debt_covered_value_eth,...,supply_withdrawal_ratio,protocol_turnover_usd,flashloan_usage_intensity,flashloan_amount_value_usd,user_activity,target_next_1d,target_fwd_max_1d,target_fwd_max_3d,target_fwd_max_7d,y_reg_log1p
352,2026-03-19 00:00:00.000 UTC,64,63,46,32,32,78105.267644,36.390256,73871.720076,34.418156,...,0.871889,2.795701e+09,0.075132,1.748341e+07,2133.0,1812.485989,1812.485989,166800.644441,3.068845e+05,7.503006
353,2026-03-20 00:00:00.000 UTC,18,17,13,9,9,1937.038887,0.908969,1812.485989,0.850522,...,1.499586,2.856165e+09,0.031188,6.974098e+06,1788.0,166800.644441,166800.644441,306884.460214,6.333518e+06,12.024561
354,2026-03-21 00:00:00.000 UTC,28,24,21,14,14,179057.447139,83.396745,166800.644441,77.688442,...,1.631851,1.491652e+09,0.042311,6.482626e+06,1495.0,9904.264475,9904.264475,306884.460214,6.333518e+06,9.200822
355,2026-03-22 00:00:00.000 UTC,16,15,10,8,8,10365.033495,5.033105,9904.264475,4.809255,...,1.034475,5.176868e+09,0.059240,1.603237e+07,1985.0,306884.460214,306884.460214,306884.460214,6.333518e+06,12.634230
356,2026-03-23 00:00:00.000 UTC,26,23,19,13,13,331174.315397,161.889194,306884.460214,150.017893,...,0.972799,5.752140e+09,0.048760,1.543877e+07,2178.0,67830.139962,67830.139962,67830.139962,6.333518e+06,11.124777


In [11]:
# target profile (info only — real thresholds are train-only, per horizon, in model_split)
target_cols = ["target_next_1d", "target_fwd_max_3d", "target_fwd_max_7d", "y_reg_log1p"]
temp_2 = adv.statistical_validation(DF_model, columns=target_cols, save=False)
display(temp_2[["column", "null_pct", "zero_pct", "mean", "cv", "p95"]])

,column,null_pct,zero_pct,mean,cv,p95
0,target_next_1d,0.0,8.6835,2.799214e+06,4.896355,6.339090e+06
1,target_fwd_max_3d,0.0,0.5602,6.731565e+06,3.241597,4.468757e+07
2,target_fwd_max_7d,0.0,0.0000,1.288218e+07,2.330667,1.150172e+08
3,y_reg_log1p,0.0,8.6835,8.995730e+00,0.534684,1.566224e+01


In [12]:
DF_model.to_csv(DATA_DIR / "DF_model_dataset_24h.csv", index=False)
print(f" wrote {len(DF_model)} rows -> {DATA_DIR}/DF_model_dataset_24h.csv")

 wrote 357 rows -> transformed_data/DF_model_dataset_24h.csv
